<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/5_joins_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("session4").config("spark.sql.shuffle.partitions", "5").getOrCreate()

In [2]:
# Employee table
emp_data = [(1, "Raj",   101),
            (2, "Priya", 102),
            (3, "Amit",  101),
            (4, "Sara",  103),
            (5, "Karan", 999)]   # 999 dept doesn't exist in dept table

emp_cols = ["emp_id", "name", "dept_id"]
emp_df = spark.createDataFrame(emp_data, emp_cols)

# Department table
dept_data = [(101, "Engineering", "Mumbai"),
             (102, "Marketing",   "Delhi"),
             (103, "HR",          "Bangalore"),
             (104, "Finance",     "Chennai")]   # no employee in Finance

dept_cols = ["dept_id", "dept_name", "location"]
dept_df = spark.createDataFrame(dept_data, dept_cols)

In [3]:
emp_df.show()

+------+-----+-------+
|emp_id| name|dept_id|
+------+-----+-------+
|     1|  Raj|    101|
|     2|Priya|    102|
|     3| Amit|    101|
|     4| Sara|    103|
|     5|Karan|    999|
+------+-----+-------+



In [4]:
dept_df.show()

+-------+-----------+---------+
|dept_id|  dept_name| location|
+-------+-----------+---------+
|    101|Engineering|   Mumbai|
|    102|  Marketing|    Delhi|
|    103|         HR|Bangalore|
|    104|    Finance|  Chennai|
+-------+-----------+---------+



In [6]:
# 1. inner join

emp_df.join(dept_df, on="dept_id", how="inner").show()    #'on=' automatically removes duplicate columns


+-------+------+-----+-----------+---------+
|dept_id|emp_id| name|  dept_name| location|
+-------+------+-----+-----------+---------+
|    102|     2|Priya|  Marketing|    Delhi|
|    101|     3| Amit|Engineering|   Mumbai|
|    101|     1|  Raj|Engineering|   Mumbai|
|    103|     4| Sara|         HR|Bangalore|
+-------+------+-----+-----------+---------+



In [11]:
#2. left join

emp_df.join(dept_df, on="dept_id", how="left").orderBy("emp_id").show()

+-------+------+-----+-----------+---------+
|dept_id|emp_id| name|  dept_name| location|
+-------+------+-----+-----------+---------+
|    101|     1|  Raj|Engineering|   Mumbai|
|    102|     2|Priya|  Marketing|    Delhi|
|    101|     3| Amit|Engineering|   Mumbai|
|    103|     4| Sara|         HR|Bangalore|
|    999|     5|Karan|       NULL|     NULL|
+-------+------+-----+-----------+---------+



In [13]:
#3 right join

emp_df.join(dept_df, on="dept_id", how="right").orderBy('emp_id').show()

+-------+------+-----+-----------+---------+
|dept_id|emp_id| name|  dept_name| location|
+-------+------+-----+-----------+---------+
|    104|  NULL| NULL|    Finance|  Chennai|
|    101|     1|  Raj|Engineering|   Mumbai|
|    102|     2|Priya|  Marketing|    Delhi|
|    101|     3| Amit|Engineering|   Mumbai|
|    103|     4| Sara|         HR|Bangalore|
+-------+------+-----+-----------+---------+



In [14]:
#4 full outer join

emp_df.join(dept_df, on="dept_id", how="full").orderBy('emp_id').show()

+-------+------+-----+-----------+---------+
|dept_id|emp_id| name|  dept_name| location|
+-------+------+-----+-----------+---------+
|    104|  NULL| NULL|    Finance|  Chennai|
|    101|     1|  Raj|Engineering|   Mumbai|
|    102|     2|Priya|  Marketing|    Delhi|
|    101|     3| Amit|Engineering|   Mumbai|
|    103|     4| Sara|         HR|Bangalore|
|    999|     5|Karan|       NULL|     NULL|
+-------+------+-----+-----------+---------+



In [23]:
#join ON multiple columns - but it needs common columns only
#emp_df.join(dept_df, on=["dept_id", "emp_id"], how="inner").show()    #since emp_id is not in dept_id so it'll throw error

In [28]:
#5 anti join - only unmatched rows from left table (returns rows from left table that are not matched in right table)

emp_df.join(dept_df, on="dept_id", how="left_anti").show()

+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|    999|     5|Karan|
+-------+------+-----+



In [29]:
#6 left semi - returns left table rows that are matched with right table , but returns only left table columns with no duplicates!
emp_df.join(dept_df, on="dept_id", how="left_semi").show()

+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|    102|     2|Priya|
|    101|     1|  Raj|
|    101|     3| Amit|
|    103|     4| Sara|
+-------+------+-----+



## **Practical task**

Find all employees with their department name and location using inner join

Find employees who don't have a valid department (hint: left join + filter nulls)

Find departments that have no employees assigned

In [31]:
emp_df.join(dept_df, on='dept_id', how="inner").show()

emp_df.join(dept_df, on='dept_id', how="left").filter(F.col('dept_name').isNull()).show()

dept_df.join(emp_df, on='dept_id', how="left").filter(F.col('name').isNull()).show()

+-------+------+-----+-----------+---------+
|dept_id|emp_id| name|  dept_name| location|
+-------+------+-----+-----------+---------+
|    102|     2|Priya|  Marketing|    Delhi|
|    101|     3| Amit|Engineering|   Mumbai|
|    101|     1|  Raj|Engineering|   Mumbai|
|    103|     4| Sara|         HR|Bangalore|
+-------+------+-----+-----------+---------+

+-------+------+-----+---------+--------+
|dept_id|emp_id| name|dept_name|location|
+-------+------+-----+---------+--------+
|    999|     5|Karan|     NULL|    NULL|
+-------+------+-----+---------+--------+

+-------+---------+--------+------+----+
|dept_id|dept_name|location|emp_id|name|
+-------+---------+--------+------+----+
|    104|  Finance| Chennai|  NULL|NULL|
+-------+---------+--------+------+----+



In [32]:
#for 2 and 3 rd question left_anti can also be used